# 01 · Activity recognition with UCI HAR

This notebook follows the data from file checks to a small model comparison. The target is one of six activities recorded while volunteers carried a waist-mounted smartphone. The table contains features calculated by the dataset authors from short sensor windows; it is not raw continuous sensor data.

We will keep UCI's participant-separated train and test sets. The test set stays untouched until the final comparison.

## 1. Load the released train and test files

Extract `UCI HAR Dataset` into `data/raw/` as described in `data/README.md`. `src/data.py` checks that the feature rows, labels, and subject IDs line up and that feature values are finite. If a file is missing, it points to the expected directory.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src" / "data.py").is_file()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.data import load_dataset
from src.models import evaluate_models

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "UCI HAR Dataset"
data = load_dataset(DATA_DIR)
train, test = data["train"], data["test"]
activity_names = data["activity_names"]
print(f"Train: {train['X'].shape[0]:,} rows; test: {test['X'].shape[0]:,} rows")
print(f"Features per row: {train['X'].shape[1]}; training participants: {len(np.unique(train['subjects']))}; test participants: {len(np.unique(test['subjects']))}")

## 2. Check labels and participants

Each row is a labeled example. The original split assigns different people to train and test, which gives a more useful check of transfer to participants the model has not seen. The next plots describe the class balance and the participant counts; they do not imply the model can recognize activities in every setting.

In [ ]:
label_rows = []
for split_name, split_data in (("train", train), ("test", test)):
    counts = pd.Series(split_data["y"]).value_counts().sort_index()
    for label, count in counts.items():
        label_rows.append({"split": split_name, "label": int(label),
                           "activity": activity_names[int(label)], "windows": int(count),
                           "participants": int(np.unique(split_data["subjects"][split_data["y"] == label]).size)})
label_summary = pd.DataFrame(label_rows)
display(label_summary)
label_summary.pivot(index="activity", columns="split", values="windows").plot.bar(figsize=(9, 4), rot=35)
plt.ylabel("Labeled windows")
plt.title("Examples in the supplied split")
plt.tight_layout()
plt.show()

## 3. Fit two reference models

The majority baseline predicts the most common training activity for every example. It gives us a simple point of comparison. The random forest can combine nonlinear patterns in the 561 supplied features. We train both on `train` and make the final predictions on `test`; we do not tune parameters using test labels.

In [ ]:
labels = sorted(activity_names)
results = evaluate_models(train["X"], train["y"], test["X"], test["y"], labels)
score_table = pd.DataFrame([{"model": name, "accuracy": result["accuracy"],
                            "macro_f1": result["macro_f1"]}
                           for name, result in results.items()])
display(score_table)

## 4. Look beyond accuracy

Accuracy is the share of test rows classified correctly. Macro F1 gives each activity equal weight, so performance on less common classes still matters. Precision asks how often a model's label is correct; recall asks how many examples of a class it recovers. The confusion matrix shows the specific mix-ups.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

rf = results["Random forest"]
report = pd.DataFrame(rf["report"]).T
display(report.loc[[activity_names[label] for label in labels], ["precision", "recall", "f1-score", "support"]].round(3))
ConfusionMatrixDisplay(rf["confusion"], display_labels=[activity_names[label] for label in labels]).plot(
    cmap="Blues", xticks_rotation=35, values_format="d")
plt.title("Random forest · supplied UCI test split")
plt.tight_layout()
plt.show()

## What this result can and cannot say

A result on UCI's test split is about these six scripted activities, this phone placement, and the authors' feature pipeline. It is not a health assessment or a measure of stress or fatigue. People, phone placement, and daily activities can differ outside this study. The later `wearable-signal-lab` project goes back to the inertial signal windows to study filtering and features directly.

### Glossary

- **Feature:** a calculated number describing part of a sensor window.
- **Label:** the activity recorded for one example.
- **Train split:** data used to fit the model.
- **Test split:** held-out data used once for the final comparison.
- **Macro F1:** average of the F1 score for each class, with equal class weight.
- **Confusion matrix:** a count of true labels versus model predictions.

### Try next

1. Compare the majority baseline with the random forest. Why is accuracy alone incomplete?
2. Find two activities with the most confusion and inspect the corresponding class-level precision and recall.
3. Compare the number of training and test examples for each activity.
4. Change one random-forest setting, such as `min_samples_leaf`, and record the change. Keep the test set for a final run; use a validation split from training data if you want to tune more carefully.
5. Read the dataset's `features_info.txt`. Choose one feature name and explain it in your own words.